### **Implementacja drzewa dwumianowego**

**Cel**: weryfikacja wyceny opcji typu *knock-and-out* uzyskanej przy użyciu metody *explicite finite difference*.

Implementacja drzewa przeprowadza wycenę niewaniliowych opcji call i put o charakterze *knock-and-out* (typu europejskiego oraz amerykańskiego) z uwzględnieniem potencjalnego wypłacenia dywidendy, będącej ustaloną wartością liczbową (niezależną od wartości aktywa bazowego w momencie wypłacenia). Drzewo (zmiany ceny aktywa bazowego oraz wycena wsteczna) korzysta ze standardowej parametryzacji CRR (Cox-Ross-Rubenstein), która odpowiada zdyskretyzowaniu modelu Blacka-Scholesa (przy $n \rightarrow \infty$ wycena zgodna z modelem BS, gdzie $n$ - liczba 'poziomów' drzewa).

Metoda uwzględnienia dywidendy: sprawdzamy, który 'poziom' drzewa (jednoznacznie powiązany z pewnym momentem czasowym) jest najbliższy chwili wypłacania dywidendy - jeśli odpowiada on faktycznej chwili wypłacania dywidendy, to dla tego poziomu obniżamy cenę aktywa bazowego o wartość dywideny; w przeciwnym wypadku, obniżamy cenę aktywa bazowego dla najbliższego poziomu, który następuje po samej chwili wypłacenia. Ze względu na uwzględnienie dywidendy, struktura cen na drzewie uniemożliwia dalsze wycenianie względem klasycznej metody (dywidenda spowodowała 'rozgałęzienie'). Wobec tego, w następnych poziomach w każdym z wierzchołków jako cenę przyjmujemy średnią arytmetyczną z cen (przemnożonych przez czynnik $u$ lub $d$) z wierzchołków połączonych z tym wierzchołkiem z poprzedniego poziomu.


#### Funkcje (wycena drzewem dwumianowym, Black-Scholes):

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy

In [ ]:
#   Funkcja zwracająca wartość opcji call / put knock-and-out typu europejskiego (z modelu Blacka-Scholesa)
def BS_knock_and_out_price(
    option_type: str,
    option_barrier: float,
    option_strike: float,
    time_to_maturity: float,
    S: float,
    volatility: float,
    risk_free_rate: float
):
    
    a = (option_barrier / S) ** (-1 + 2 * risk_free_rate / volatility ** 2)
    b = (option_barrier / S) ** (1 + 2 * risk_free_rate / volatility ** 2)
    
    d1 = (np.log(S / option_strike) + (risk_free_rate + 1/2 * volatility**2) * time_to_maturity) / (volatility * np.sqrt(time_to_maturity))
    d2 = (np.log(S / option_strike) + (risk_free_rate - 1/2 * volatility**2) * time_to_maturity) / (volatility * np.sqrt(time_to_maturity))
    d3 = (np.log(S / option_barrier) + (risk_free_rate + 1/2 * volatility**2) * time_to_maturity) / (volatility * np.sqrt(time_to_maturity))
    d4 = (np.log(S / option_barrier) + (risk_free_rate - 1/2 * volatility**2) * time_to_maturity) / (volatility * np.sqrt(time_to_maturity))
    d5 = (np.log(S / option_barrier) - (risk_free_rate - 1/2 * volatility**2) * time_to_maturity) / (volatility * np.sqrt(time_to_maturity))
    d6 = (np.log(S / option_barrier) - (risk_free_rate + 1/2 * volatility**2) * time_to_maturity) / (volatility * np.sqrt(time_to_maturity))
    d7 = (np.log(S * option_strike / option_barrier**2) - (risk_free_rate - 1/2 * volatility**2) * time_to_maturity) / (volatility * np.sqrt(time_to_maturity))
    d8 = (np.log(S * option_strike / option_barrier**2) - (risk_free_rate + 1/2 * volatility**2) * time_to_maturity) / (volatility * np.sqrt(time_to_maturity))
    
    if option_type == 'call':
        return(
            S * (scipy.stats.norm.cdf(d1) - scipy.stats.norm.cdf(d3) - b * (scipy.stats.norm.cdf(d6) - scipy.stats.norm.cdf(d8))) -
            option_strike * np.exp(-risk_free_rate * time_to_maturity) * (scipy.stats.norm.cdf(d2) - scipy.stats.norm.cdf(d4) -
            a * (scipy.stats.norm.cdf(d5) - scipy.stats.norm.cdf(d7)))
        )
    elif option_type == 'put':
        return(
            -S * (scipy.stats.norm.cdf(d3) - scipy.stats.norm.cdf(d1) - b * (scipy.stats.norm.cdf(d8) - scipy.stats.norm.cdf(d6))) +
            option_strike * np.exp(-risk_free_rate * time_to_maturity) * (scipy.stats.norm.cdf(d4) - scipy.stats.norm.cdf(d2) -
            a * (scipy.stats.norm.cdf(d7) - scipy.stats.norm.cdf(d5)))
        )

In [ ]:
#   Funkcja przeprowadzająca wycenę opcji call / put knock-and-out typu europejskiego / amerykańskiego według metody
#   drzewa dwumianowego. Zwraca trzy obiekty, kolejno:
#   (1) wartość opcji na chwilę zero
#   (2) siatkę (macierz rozmiaru (steps+1)^2) wartości opcji na drzewie
#   (3) siatkę binarną optymalnych momentów wykonania opcji (dla opcji typu amerykańskiego)
def knock_and_out_option_price(
    option_type: str,
    option_style: str,
    option_barrier: float,
    option_strike: float,
    dividend_value: float,
    dividend_time: float,
    time_to_maturity: float,
    S0: float,
    volatility: float,
    risk_free_rate: float,
    steps: int    
):
    
    time_step = time_to_maturity / steps
    
    u = np.exp(volatility * np.sqrt(time_step))
    d = np.exp( - volatility * np.sqrt(time_step))
    
    risk_free_probability = (np.exp(risk_free_rate * time_step) - d) / (u - d)
    discount_factor = np.exp( - risk_free_rate * time_step)   
    
    optimal_exercise_times = np.zeros((steps + 1)**2).reshape(steps + 1, steps + 1)
    option_values_lattice = np.zeros((steps + 1)**2).reshape(steps + 1, steps + 1)
    
    u_lattice = np.zeros((steps + 1)**2).reshape(steps + 1, steps + 1)
    d_lattice = np.zeros((steps + 1)**2).reshape(steps + 1, steps + 1)
    
    for i in range(steps + 1):
        u_lattice[i,i:] = np.arange(steps + 1 - i)
        d_lattice[i,:] = np.concat((np.zeros(i), np.arange(i, steps + 1)))
        
    d_lattice = d ** (d_lattice - u_lattice)
    u_lattice = u ** u_lattice

    if dividend_value == 0:
        price_lattice = S0 * u_lattice * d_lattice
        
    else:
        moments = np.arange(0, steps + 1) * time_to_maturity / steps
        dividend_level = np.where(moments >= dividend_time)[0][0]       
        price_lattice = np.zeros((steps + 1)**2).reshape(steps + 1, steps + 1)
        price_lattice[:, :dividend_level + 1] = (S0 * u_lattice * d_lattice)[:, :dividend_level + 1]
        price_lattice[:dividend_level + 1, dividend_level] = price_lattice[:dividend_level + 1, dividend_level] - dividend_value

        for i in range(dividend_level + 1, steps + 1):
            price_lattice[0, i] = price_lattice[0, i - 1] * u
            price_lattice[i, i] = price_lattice[i - 1, i - 1] * d
            for j in range(1, i):
                price_lattice[j, i] = 1/2 * (price_lattice[j - 1, i - 1] * d + price_lattice[j, i - 1] * u)
                           
    
    
    if option_type == 'call':
        barrier_memory_lattice = (price_lattice <= option_barrier)
    elif option_type == 'put':
        barrier_memory_lattice = (price_lattice >= option_barrier)
    else:
        raise NotImplementedError(
            f"{option_type} is not a valid option type (only call/put permited)"
        )

    for i in range(steps, -1, -1):
        for j in range(0, i + 1):
            if i == steps:
                if option_type == 'call':
                    option_values_lattice[j, i] = max(price_lattice[j, i] - option_strike, 0) * barrier_memory_lattice[j, i]
                elif option_type == 'put':
                    option_values_lattice[j, i] = max(option_strike - price_lattice[j, i], 0) * barrier_memory_lattice[j, i]
                else:
                    raise NotImplementedError(
                        f"{option_type} is not a valid option type (only call/put permited)"
                    )
            else:
                option_value = (option_values_lattice[j, i + 1] * risk_free_probability + 
                                option_values_lattice[j + 1, i + 1] * (1 - risk_free_probability)
                        ) * discount_factor
                
                if option_style == 'european':
                    option_values_lattice[j, i] = option_value * barrier_memory_lattice[j, i] 
                    
                elif option_style == 'american':
                    if option_type == 'call':
                        if option_value <= max(price_lattice[j, i] - option_strike, 0):
                            optimal_exercise_times[j, i] = 1 * barrier_memory_lattice[j, i]
                            option_values_lattice[j, i] = max(price_lattice[j, i] - option_strike, 0) * barrier_memory_lattice[j, i]
                        else:
                            option_values_lattice[j, i] = option_value * barrier_memory_lattice[j, i]
                    elif option_type == 'put':
                        if option_value <= max(option_strike - price_lattice[j, i], 0):
                            optimal_exercise_times[j, i] = 1 * barrier_memory_lattice[j, i]
                            option_values_lattice[j, i] = max(option_strike - price_lattice[j, i], 0) * barrier_memory_lattice[j, i]
                        else:
                            option_values_lattice[j, i] = option_value * barrier_memory_lattice[j, i]
                    else:
                        raise NotImplementedError(
                            f"{option_type} is not a valid option type (only call/put permited)"
                        )
                else:
                    raise NotImplementedError(
                        f"{option_style} is not a valid option style (only european/american permited)"
                    )
                    
    option_price = option_values_lattice[0, 0]

    return(option_price, option_values_lattice, optimal_exercise_times)      

**Parametry opcji z projektu:**

$S0 = 3200$ (przybliżona cena indeksu WIG20 z początku roku 2026),

$K = 3200$ (cena wykonania zarówno opcji call i put),

$B_C = 3600$ (bariera dla *knock-and-out* call),

$B_P = 2800$ (bariera dla *knock-and-out* put),

$T = 1$.

Roboczo przyjmujemy, że $r = 0.05$ oraz $\sigma = 0.2$.
W przypadku gdy uwzględniamy wypłatę dywidendy: wynosi ona $q = 300$, a wypłata następuje w chwili $t = 1/2$.

#### Wyliczanie wartości opcji:

In [75]:
call_no_div_europ_value, emp, emp = knock_and_out_option_price(
    option_type= 'call',
    option_style= 'european',
    option_barrier= 3600,
    option_strike= 3200,
    dividend_value= 0,
    dividend_time= 0,
    time_to_maturity= 1,
    S0= 3200,
    volatility= 0.2,
    risk_free_rate= 0.05,
    steps = 10000
)
put_no_div_europ_value, emp, emp = knock_and_out_option_price(
    option_type= 'put',
    option_style= 'european',
    option_barrier= 2800,
    option_strike= 3200,
    dividend_value= 0,
    dividend_time= 0,
    time_to_maturity= 1,
    S0= 3200,
    volatility= 0.2,
    risk_free_rate= 0.05,
    steps = 10000
)

In [81]:
call_no_div_amer_value, emp, emp = knock_and_out_option_price(
    option_type= 'call',
    option_style= 'american',
    option_barrier= 3600,
    option_strike= 3200,
    dividend_value= 0,
    dividend_time= 0,
    time_to_maturity= 1,
    S0= 3200,
    volatility= 0.2,
    risk_free_rate= 0.05,
    steps = 10000
)
put_no_div_amer_value, emp, emp = knock_and_out_option_price(
    option_type= 'put',
    option_style= 'american',
    option_barrier= 2800,
    option_strike= 3200,
    dividend_value= 0,
    dividend_time= 0,
    time_to_maturity= 1,
    S0= 3200,
    volatility= 0.2,
    risk_free_rate= 0.05,
    steps = 10000
)

In [83]:
call_div_europ_value, emp, emp = knock_and_out_option_price(
    option_type= 'call',
    option_style= 'european',
    option_barrier= 3600,
    option_strike= 3200,
    dividend_value= 300,
    dividend_time= 1/2,
    time_to_maturity= 1,
    S0= 3200,
    volatility= 0.2,
    risk_free_rate= 0.05,
    steps = 10000
)
put_div_europ_value, emp, emp = knock_and_out_option_price(
    option_type= 'put',
    option_style= 'european',
    option_barrier= 2800,
    option_strike= 3200,
    dividend_value= 300,
    dividend_time= 1/2,
    time_to_maturity= 1,
    S0= 3200,
    volatility= 0.2,
    risk_free_rate= 0.05,
    steps = 10000
)

In [85]:
call_div_amer_value, emp, emp = knock_and_out_option_price(
    option_type= 'call',
    option_style= 'american',
    option_barrier= 3600,
    option_strike= 3200,
    dividend_value= 300,
    dividend_time= 1/2,
    time_to_maturity= 1,
    S0= 3200,
    volatility= 0.2,
    risk_free_rate= 0.05,
    steps = 10000
)
put_div_amer_value, emp, emp = knock_and_out_option_price(
    option_type= 'put',
    option_style= 'american',
    option_barrier= 2800,
    option_strike= 3200,
    dividend_value= 300,
    dividend_time= 1/2,
    time_to_maturity= 1,
    S0= 3200,
    volatility= 0.2,
    risk_free_rate= 0.05,
    steps = 10000
)

In [77]:
BS_barrier_call = BS_knock_and_out_price(
    option_type= 'call',
    option_barrier= 3600,
    option_strike= 3200,
    time_to_maturity= 1,
    S = 3200,
    volatility= 0.2,
    risk_free_rate= 0.05
)
BS_barrier_put = BS_knock_and_out_price(
    option_type= 'put',
    option_barrier= 2800,
    option_strike= 3200,
    time_to_maturity= 1,
    S = 3200,
    volatility= 0.2,
    risk_free_rate= 0.05
)

#### Wyniki:

In [87]:
print('Knock-and-out European Call Price, no dividend (Binomial tree method): ', np.round(call_no_div_europ_value, 6))
print('Knock-and-out European Call Price, no dividend (Black-Scholes formula): ', np.round(BS_barrier_call, 6))
print('Knock-and-out European Put Price, no dividend (Binomial tree method): ', np.round(put_no_div_europ_value, 6))
print('Knock-and-out European Put Price, no dividend (Black-Scholes formula): ', np.round(BS_barrier_put, 6))
print('---')
print('Knock-and-out American Call Price, no dividend (Binomial tree method): ', np.round(call_no_div_amer_value, 6))
print('Knock-and-out American Put Price, no dividend (Binomial tree method): ', np.round(put_no_div_amer_value, 6))
print('---')
print('Knock-and-out European Call Price, dividend included (Binomial tree method): ', np.round(call_div_europ_value, 6))
print('Knock-and-out European Put Price, dividend included (Binomial tree method): ', np.round(put_div_europ_value, 6))
print('Knock-and-out American Call Price, dividend included (Binomial tree method): ', np.round(call_div_amer_value, 6))
print('Knock-and-out American Put Price, dividend included (Binomial tree method): ', np.round(put_div_amer_value, 6))



Knock-and-out European Call Price, no dividend (Binomial tree method):  8.397123
Knock-and-out European Call Price, no dividend (Black-Scholes formula):  8.342918
Knock-and-out European Put Price, no dividend (Binomial tree method):  11.192752
Knock-and-out European Put Price, no dividend (Black-Scholes formula):  11.064386
---
Knock-and-out American Call Price, no dividend (Binomial tree method):  243.92455
Knock-and-out American Put Price, no dividend (Binomial tree method):  189.10456
---
Knock-and-out European Call Price, dividend included (Binomial tree method):  6.098286
Knock-and-out European Put Price, dividend included (Binomial tree method):  8.351775
Knock-and-out American Call Price, dividend included (Binomial tree method):  209.649436
Knock-and-out American Put Price, dividend included (Binomial tree method):  240.499521
